In [15]:
import pandas as pd
from collections import defaultdict

In [16]:
# 1. DEFINE THE FP-TREE NODE STRUCTURE
class FPTreeNode:
    def __init__(self, item, count=0, parent=None):
        self.item = item          # The name of the item (e.g., 'Item_Blouse')
        self.count = count        # How many times this item appears in this path
        self.parent = parent      # Reference to the parent node
        self.children = {}        # Dictionary to store child nodes
        self.next = None          # Link to the next node of the same item type

    def increment(self, count):
        #Increases the counter for this node
        self.count += count

    def display(self, indent=1):
        #Recursively prints the tree structure visually
        print('  ' * indent + f"{self.item} ({self.count})")
        for child in self.children.values():
            child.display(indent + 1)


# 2. FUNCTION TO BUILD THE FP-TREE

def construct_fp_tree(transactions, min_support):
    #Creates the tree and the header table from the list of transactions

    # --- Step A: Count how often every single item appears ---
    item_counts = defaultdict(int)
    for transaction in transactions:
        for item in transaction:
            item_counts[item] += 1

    # --- Step B: Filter out items that don't meet the minimum support ---
    frequent_items = {item: count for item, count in item_counts.items() if count >= min_support}

    # If no items are frequent enough, we stop here
    if not frequent_items:
        return None, None

    # --- Step C: Re-order transactions based on item frequency ---
    ordered_transactions = []
    for transaction in transactions:
        # Keep only the frequent items from the original transaction
        ordered_items = [item for item in transaction if item in frequent_items]
        # Sort items: most popular first, then alphabetically
        ordered_items.sort(key=lambda item: (-frequent_items[item], item))
        if ordered_items:
            ordered_transactions.append(ordered_items)

    # --- Step D: Insert transactions into the tree ---
    root = FPTreeNode(None, 1, None) # The root starts as 'None'
    header_table = defaultdict(list)

    for transaction in ordered_transactions:
        current_node = root
        for item in transaction:
            # If child doesn't exist, create a new node
            if item not in current_node.children:
                new_node = FPTreeNode(item, 1, current_node)
                current_node.children[item] = new_node
                header_table[item].append(new_node) # Track node in header table
            else:
                # If child exists, just increase its count
                current_node.children[item].increment(1)
            # Move down the tree to the next item
            current_node = current_node.children[item]

    return root, header_table


# 3. FUNCTION TO MINE FREQUENT PATTERNS

def mine_fp_tree(header_table, min_support, prefix=None, frequent_patterns=None):
    #Finds all combinations of items that appear frequently together
    if prefix is None:
        prefix = []
    if frequent_patterns is None:
        frequent_patterns = {}

    # Sort items in the header table (bottom-up approach)
    for item, nodes in sorted(header_table.items(), key=lambda x: x[0], reverse=True):
        new_prefix = prefix + [item]
        # Support is the sum of counts across all nodes of this item
        support = sum(node.count for node in nodes)

        if support >= min_support:
            frequent_patterns[frozenset(new_prefix)] = support

        # Find the conditional paths (all paths leading to this item)
        conditional_transactions = []
        for node in nodes:
            path = []
            parent = node.parent
            while parent and parent.item is not None:
                path.append(parent.item)
                parent = parent.parent
            path.reverse()
            # Add this path to our "mini-dataset" for the next recursion
            if path:
                conditional_transactions.extend([path] * node.count)

        # Build a "conditional tree" for this specific item
        conditional_tree, conditional_header = construct_fp_tree(conditional_transactions, min_support)
        if conditional_tree is not None:
            # Recursively mine the smaller tree
            mine_fp_tree(conditional_header, min_support, new_prefix, frequent_patterns)

    return frequent_patterns


In [17]:
# 4. DATA PREPARATION
# Read the CSV file
data = pd.read_csv("C:/Users/asus/Downloads/shopping_behavior_updated (1).csv")

# Define which original columns to include and the prefix for each
columns_to_use = [
    ('Item Purchased', 'Item'),
    ('Color', 'Color'),
    ('Season', 'Season'),
    ('Size', 'Size'),

]

# Create a new column for each with the desired prefix
for col_name, prefix in columns_to_use:
    data[f'Item_{col_name}'] = prefix + '_' + data[col_name].astype(str)

# Build a list of all the new column names
item_columns = [f'Item_{col_name}' for col_name, _ in columns_to_use]

# Convert the selected columns to a list of transactions
transactions = data[item_columns].values.tolist()
# Set the threshold (A pattern must appear at least 350 times to be 'frequent')
min_support_count = 350

# Build the tree
fp_tree, header_table = construct_fp_tree(transactions, min_support_count)

if fp_tree is not None:
    # Print the tree structure
    print("FP-Tree Structure:")
    fp_tree.display()

    # Mine the patterns
    print("\nFrequent Patterns:")
    frequent_patterns = mine_fp_tree(header_table, min_support_count)

    # Sort and print the patterns in the requested format
    for pattern, support in sorted(frequent_patterns.items(), key=lambda x: x[1], reverse=True):
        pattern_str = ", ".join([f"'{item}'" for item in sorted(list(pattern))])
        print(f"{{{pattern_str}}}: {support}")
else:
    print("No frequent itemsets found.")

FP-Tree Structure:
  None (1)
    Size_L (1053)
      Season_Winter (280)
      Season_Summer (257)
      Season_Fall (260)
      Season_Spring (256)
    Season_Spring (291)
      Size_S (179)
      Size_XL (112)
    Size_M (1755)
      Season_Spring (452)
      Season_Summer (445)
      Season_Fall (439)
      Season_Winter (419)
    Season_Winter (272)
      Size_S (152)
      Size_XL (120)
    Season_Summer (253)
      Size_S (160)
      Size_XL (93)
    Season_Fall (276)
      Size_S (172)
      Size_XL (104)

Frequent Patterns:
{'Size_M'}: 1755
{'Size_L'}: 1053
{'Season_Spring'}: 999
{'Season_Fall'}: 975
{'Season_Winter'}: 971
{'Season_Summer'}: 955
{'Size_S'}: 663
{'Season_Spring', 'Size_M'}: 452
{'Season_Summer', 'Size_M'}: 445
{'Season_Fall', 'Size_M'}: 439
{'Size_XL'}: 429
{'Season_Winter', 'Size_M'}: 419


In [18]:
# Define the columns for the new tree (Category and Gender)
new_columns = [
    ('Category', 'Category'),
    ('Gender', 'Gender')
]
for col_name, prefix in new_columns:
    data[f'Item_{col_name}'] = prefix + '_' + data[col_name].astype(str)

# Build the transaction list using only these new columns
new_item_columns = [f'Item_{col_name}' for col_name, _ in new_columns]
new_transactions = data[new_item_columns].values.tolist()

# min_support = int(0.1 * len(new_transactions))   # 10% of total transactions
min_support = 500   # adjust as needed

# Build the new FP-tree
new_tree, new_header = construct_fp_tree(new_transactions, min_support)

if new_tree is not None:

    print("FP-Tree for Category & Gender")

    new_tree.display()

    print("\nFrequent Patterns (Category & Gender):")
    new_patterns = mine_fp_tree(new_header, min_support)
    for pattern, support in sorted(new_patterns.items(), key=lambda x: x[1], reverse=True):
        print(f"{set(pattern)}: {support}")
else:
    print("No frequent itemsets found for Category & Gender.")

FP-Tree for Category & Gender
  None (1)
    Gender_Male (2652)
      Category_Clothing (1181)
      Category_Footwear (400)
      Category_Accessories (848)
    Category_Clothing (556)
      Gender_Female (556)
    Gender_Female (692)
      Category_Accessories (392)
      Category_Footwear (199)

Frequent Patterns (Category & Gender):
{'Gender_Male'}: 2652
{'Category_Clothing'}: 1737
{'Gender_Female'}: 1248
{'Category_Accessories'}: 1240
{'Category_Clothing', 'Gender_Male'}: 1181
{'Category_Accessories', 'Gender_Male'}: 848
{'Category_Footwear'}: 599
{'Category_Clothing', 'Gender_Female'}: 556
